In [1]:
%load_ext autoreload
%autoreload 2

from Shared.shared import *
from Shared.specific_CNB_sim import *

sim_name = f"SunMod_2k"
sim_folder = f"sim_output/{sim_name}"
fig_folder = f"figures_local/{sim_name}"
Cl_folder = f"Shared/Cls"
nu_m_range = jnp.load(f"{sim_folder}/neutrino_massrange_eV.npy")
nu_m_picks = jnp.array([0.01, 0.05, 0.1, 0.2, 0.3])*Params.eV
simdata = SimData(sim_folder)

In [4]:

def inverse_parent_momentum_costh(p_d, costh, M, m):
    """
    Computes the parent’s momentum p_N in the inverse reaction:
      daughter(m, momentum p_d along +x-axis) + massless scalar -> parent(M)
    The parent emerges at angle theta with cos(theta) = costh.
    Returns None if there’s no physically valid (positive) solution.
    Parameters:
      p_d   : float, daughter momentum magnitude (along x-axis)
      costh : float, cos(theta) for the parent emission angle
      M     : float, parent mass
      m     : float, daughter mass
    Output:
      p_N   : float or None
    """
    
    # We’ll define a few intermediate variables:
    E_nu = math.sqrt(m**2 + p_d**2)  # daughter energy
    # Our final parent’s energy: E_N = sqrt(M^2 + p_N^2)
    # We’ll get p_N by solving the polynomial:
    # Expand the polynomial (M^2 + p_N^2)(m^2 + p_d^2)
    # - [p_N p_d costh + (M^2 + m^2)/2]^2 = 0
    #
    # That yields A p_N^2 + B p_N + C = 0 with:
    # A, B, C can be derived as:
    # A = m^2 + p_d^2 - p_d^2 * costh^2 + ...
    # but let’s do a direct approach:
    # We’ll do the direct polynomial approach:
    # Let:
    #  LHS = (M^2 + pN^2)(m^2 + p_d^2)
    #        - [pN*p_d*costh + 0.5*(M^2 + m^2)]^2
    # We want LHS = 0. Then we do LHS as function of pN => polynomial in pN.
    # define a small function for LHS in terms of pN:
    def lhs(pN):
        term1 = (M**2 + pN**2)*(m**2 + p_d**2)
        term2 = (pN*p_d*costh + 0.5*(M**2 + m**2))**2
        return term1 - term2
    # We’ll convert lhs=0 to A*pN^2 + B*pN + C=0 by expanding.
    # Let’s do it systematically:
    # Expand (M^2 + pN^2)(m^2 + p_d^2) = (M^2)(m^2 + p_d^2) + pN^2(m^2 + p_d^2)
    # Expand [pN*p_d*costh + 0.5*(M^2 + m^2)]^2
    #   = pN^2*p_d^2*costh^2 + pN*p_d*costh*(M^2 + m^2) + 0.25*(M^2 + m^2)^2
    # So LHS =
    #   [M^2(m^2 + p_d^2)] + [pN^2(m^2 + p_d^2)]
    # - [pN^2 p_d^2 costh^2 + pN p_d costh (M^2 + m^2) + 0.25*(M^2+m^2)^2 ]
    # group terms in pN^2, pN, constant:
    A = (m**2 + p_d**2) - (p_d**2)*(costh**2)
    B = - (p_d*costh)*(M**2 + m**2)
    C = (M**2)*(m**2 + p_d**2) - 0.25*(M**2 + m**2)**2
    # Quadratic eq: A pN^2 + B pN + C = 0.
    disc = B**2 - 4.0*A*C
    if disc < 0:
        return None
    sqrt_disc = math.sqrt(disc)
    sol1 = (-B + sqrt_disc)/(2.0*A)
    sol2 = (-B - sqrt_disc)/(2.0*A)
    candidates = [p for p in (sol1, sol2) if p>0]
    if not candidates:
        return None
    return min(candidates)


# Example parameters
p_d_test = 1.0    # daughter momentum

costh_test = 0.5  # cos(theta)
M_test = 0.1     # parent mass
m_test = 0.05     # daughter mass
p_N_result = inverse_parent_momentum_costh(p_d_test, costh_test, M_test, m_test)
print(f"Computed parent momentum p_N =", p_N_result)

Computed parent momentum p_N = None
